# **Construct payoff matrix with given companies data** (7.1)

In [11]:
import numpy as np
import pandas as pd

N = 3
prices = np.array([10, 8, 6, 4, 2])
cost_company_1 = np.array([5, 4, 3 + 0.1 * N, 2, 1.5 - 0.1 * N])
cost_company_2 = np.array([8, 6, 4 - 0.2 * N, 2, 1 + 0.1 * N])


shares_data = {
    (10, 10): 0.31,
    (10, 8): 0.33,
    (10, 6): 0.25,
    (10, 4): 0.2,
    (10, 2): 0.18,
    (8, 10): 0.4,
    (8, 8): 0.35,
    (8, 6): 0.32,
    (8, 4): 0.28,
    (8, 2): 0.25,
    (6, 10): 0.52,
    (6, 8): 0.48,
    (6, 6): 0.4,
    (6, 4): 0.35,
    (6, 2): 0.3,
    (4, 10): 0.6,
    (4, 8): 0.58,
    (4, 6): 0.55,
    (4, 4): 0.5,
    (4, 2): 0.4,
    (2, 10): 0.9,
    (2, 8): 0.85,
    (2, 6): 0.7,
    (2, 4): 0.65,
    (2, 2): 0.4,
}

market_volume_f = lambda p1, p2:  8 - 0.3 * ((p1 + p2) / 2)

In [12]:
def create_payoff_matrix(prices, costs1, costs2, shares_dict, market_volume_f, techs_count=5):
    """
    Generates a payoff matrix based on the difference between profits
    of Company 1 and Company 2.
    """

    matrix = np.zeros((techs_count, techs_count))

    for i in range(techs_count):  # Strategy for Company 1
        for j in range(techs_count):  # Strategy for Company 2
            p1, p2 = prices[i], prices[j]

            # Market parameters
            market_volume = market_volume_f(p1, p2)
            share_p1 = shares_dict[(p1, p2)]

            # Costs for selected technologies
            c1 = costs1[i]
            c2 = costs2[j]  # Fixed index to 'j' for the 2nd company

            # Calculate individual profits
            profit1 = share_p1 * market_volume * (p1 - c1)
            profit2 = (1 - share_p1) * market_volume * (p2 - c2)

            # Payoff is the difference (D = P1 - P2)
            matrix[i, j] = round(profit1 - profit2, 2)

    # Create labeled DataFrame
    labels = [t + 1 for t in range(techs_count)]
    df = pd.DataFrame(matrix, index=labels, columns=labels)
    df.index.name = "1st Company"
    df.columns.name = "2nd Company"

    return df

df_game = create_payoff_matrix(
    prices, cost_company_1, cost_company_2, shares_data, market_volume_f
)
# Display styled result
display(
    df_game.style.set_caption("Payoff Matrix (Difference in Profits)").format(
        precision=2
    )
)

2nd Company,1,2,3,4,5
1st Company,,,,,
1,0.85,1.64,-3.92,-3.54,2.02
2,2.12,0.56,-2.88,-1.98,3.09
3,2.49,1.51,-2.98,-2.31,2.18
4,2.36,1.98,-0.45,0.00,2.70
5,3.22,2.47,-1.50,-1.28,-0.74


# Reduce equal or dominated strategies for both companies

In [13]:
def get_reduced_matrix(df):
    """
    Creates a copy of the payoff matrix and removes dominated or duplicate strategies
    for both companies. Returns a clean, reduced DataFrame.
    """
    # Create a copy to avoid modifying the original data
    reduced_df = df.copy()

    # 1. Cleaning Company 1 strategies (Rows)
    to_remove_rows = []
    indices = reduced_df.index.tolist()

    for i in indices:
        for j in indices:
            if i == j or i in to_remove_rows:
                continue

            # Check for Domination (i is worse than j) or Duplication (i equals j)
            is_dominated = all(reduced_df.loc[j] >= reduced_df.loc[i]) and any(
                reduced_df.loc[j] > reduced_df.loc[i]
            )
            is_duplicate = (
                all(reduced_df.loc[j] == reduced_df.loc[i]) and j < i
            )  # Keep only the first occurrence

            if is_dominated or is_duplicate:
                to_remove_rows.append(i)
                reason = "dominated" if is_dominated else "duplicate"
                print(f"Removing Company 1 strategy {i} ({reason} by {j})")
                break

    reduced_df.drop(index=to_remove_rows, inplace=True)

    # 2. Cleaning Company 2 strategies (Columns)
    to_remove_cols = []
    cols = reduced_df.columns.tolist()

    for i in cols:
        for j in cols:
            if i == j or i in to_remove_cols:
                continue

            # For Player 2: j dominates i if j results in LOWER payoffs for Player 1
            is_dominated = all(reduced_df[j] <= reduced_df[i]) and any(
                reduced_df[j] < reduced_df[i]
            )
            is_duplicate = all(reduced_df[j] == reduced_df[i]) and j < i

            if is_dominated or is_duplicate:
                to_remove_cols.append(i)
                reason = "dominated" if is_dominated else "duplicate"
                print(f"Removing Company 2 strategy {i} ({reason} by {j})")
                break

    reduced_df.drop(columns=to_remove_cols, inplace=True)

    return reduced_df

df_reduced = get_reduced_matrix(df_game)

df_styled = df_reduced.style.set_caption("Reduced payoff matrix").format(precision=2)
display(df_reduced)

Removing Company 1 strategy 1 (dominated by 4)
Removing Company 2 strategy 1 (dominated by 2)
Removing Company 2 strategy 2 (dominated by 3)
Removing Company 2 strategy 4 (dominated by 3)
Removing Company 2 strategy 5 (dominated by 3)


2nd Company,3
1st Company,
2,-2.88
3,-2.98
4,-0.45
5,-1.50


# Check for saddle point

Find Maximin for the 1st company. The greatest profit for 2nd company is the minimal profit for the 1st company, so 1st company cannot risk.

In [14]:
def find_equilibrium(df):
    """
    Finds the saddle point (equilibrium) of a payoff matrix.
    Calculates Alpha (maximin) and Beta (minimax) and checks if they match.
    """
    # 1. Calculate Alpha (Lower value of the game)
    row_minima = df.min(axis=1)
    alpha = row_minima.max()
    p1_strategy = row_minima.idxmax()

    # 2. Calculate Beta (Upper value of the game)
    col_maxima = df.max(axis=0)
    beta = col_maxima.min()
    p2_strategy = col_maxima.idxmin()

    # 3. Check for the existence of a saddle point
    is_saddle_point = alpha == beta

    print(f"Alpha (Maximin): {alpha:.2f}")
    print(f"Beta (Minimax): {beta:.2f}")

    if is_saddle_point:
        print(f"STATUS: Saddle point FOUND at Strategy [{p1_strategy}; {p2_strategy}]")
    else:
        print("STATUS: No saddle point found. Mixed strategies required.")

    # Return results for further automated use
    return {
        "is_saddle": is_saddle_point,
        "value": alpha if is_saddle_point else None,
        "maximin_strat": p1_strategy,
        "minimax_strat": p2_strategy,
    }

saddle_res = find_equilibrium(df_reduced)

Alpha (Maximin): -0.45
Beta (Minimax): -0.45
STATUS: Saddle point FOUND at Strategy [4; 3]


# Make a decision about the winning company at the saddle point

In [15]:
# 1. Get strategies from your function result
# p1_strat and p2_strat are the tech numbers (e.g., 4 or 5)
bot_strat = saddle_res["maximin_strat"]
up_strat = saddle_res["minimax_strat"]
game_value = saddle_res["value"]

# 2. Map strategies to actual prices
p1_eq = prices[bot_strat - 1]
p2_eq = prices[up_strat - 1]

# 3. Calculate market metrics
total_volume_eq = market_volume_f(p1_eq, p2_eq)
share1_eq = shares_data[(p1_eq, p2_eq)]
share2_eq = 1 - share1_eq

# 4. Calculate individual profits for verification
# Using equilibrium technology costs
c1_eq = cost_company_1[bot_strat - 1]
c2_eq = cost_company_2[up_strat - 1]

profit1_eq = (p1_eq - c1_eq) * total_volume_eq * share1_eq
profit2_eq = (p2_eq - c2_eq) * total_volume_eq * share2_eq

# 5. Verification: Difference should match the game value (Alpha/Beta)
calculated_diff = profit1_eq - profit2_eq

print(f"Equilibrium Final Results (Prices: {p1_eq} vs {p2_eq})")
print(f"Total market volume (S): {total_volume_eq:.2f}")
print(f"Company 1: Share = {share1_eq:.2%}, Profit = {profit1_eq:.2f}")
print(f"Company 2: Share = {share2_eq:.2%}, Profit = {profit2_eq:.2f}")
print("-" * 40)
print(f"Difference (P1 - P2): {calculated_diff:.2f}")
print(f"Game value from Matrix: {game_value:.2f}")

# Final conclusion
winner = "Company 1" if profit1_eq > profit2_eq else "Company 2"
print(f"\nWinning side: {winner}")

if abs(calculated_diff - game_value) < 0.01:
    print("SUCCESS: The calculated profit difference matches the game theory results.")
else:
    print("WARNING: Difference mismatch. Check matrix calculation logic.")

Equilibrium Final Results (Prices: 4 vs 6)
Total market volume (S): 6.50
Company 1: Share = 55.00%, Profit = 7.15
Company 2: Share = 45.00%, Profit = 7.61
----------------------------------------
Difference (P1 - P2): -0.46
Game value from Matrix: -0.45

Winning side: Company 2
SUCCESS: The calculated profit difference matches the game theory results.


# Check for another companies data (7.2)

In [22]:
N = 14
prices_2 = np.array([10, 8, 6, 4, 2])
cost_company_1_2 = np.array(
    [
        5, 
        4 - 0.1 * N, 
        3 + 0.1 * N, 
        2, 
        1.5 - 0.1 * N, 
    ]
)
cost_company_2_2 = np.array(
    [
        8,  
        6,  
        4 - 0.2 * N, 
        2,  
        1 + 0.1 * N,  
    ]
)

shares_data_2 = {
    (10, 10): 0.31 + 0.1 * (N - 1), 
    (10, 8): 0.33,
    (10, 6): 0.25,
    (10, 4): 0.2,
    (10, 2): 0.18,
    (8, 10): 0.4,
    (8, 8): 0.35,
    (8, 6): 0.32,
    (8, 4): 0.28,
    (8, 2): 0.25,
    (6, 10): 0.52,
    (6, 8): 0.48,
    (6, 6): 0.4,
    (6, 4): 0.35,
    (6, 2): 0.3 - 0.02 * N, 
    (4, 10): 0.6,
    (4, 8): 0.58,
    (4, 6): 0.55 + 0.05 * N,  
    (4, 4): 0.5,
    (4, 2): 0.4,
    (2, 10): 0.9,
    (2, 8): 0.85,
    (2, 6): 0.7,
    (2, 4): 0.65,
    (2, 2): 0.4,
}


market_volume_f_2 = lambda p1, p2: 8 - (0.3 + 0.1 * (N - 1)) * ((p1 + p2) / 2)

# Construct payoff matrix

In [23]:
df_game_2 = create_payoff_matrix(
    prices_2, cost_company_1_2, cost_company_2_2, shares_data_2, market_volume_f_2
)
# Display styled result
display(
    df_game_2.style.set_caption("Payoff Matrix (Difference in Profits)").format(
        precision=2
    )
)

2nd Company,1,2,3,4,5
1st Company,,,,,
1,-74.16,-1.98,11.28,1.92,-1.96
2,-6.14,-2.83,4.92,-0.12,0.00
3,0.61,0.87,3.58,0.00,0.68
4,-1.28,-0.51,0.00,0.00,3.33
5,-2.42,0.00,-0.18,1.71,4.80


# Reduce matrix

In [24]:
df_reduced_2 = get_reduced_matrix(df_game_2)

df_styled = df_reduced_2.style.set_caption("Reduced payoff matrix for 2nd dataset").format(precision=2)
display(df_styled)

Removing Company 2 strategy 2 (dominated by 1)
Removing Company 2 strategy 3 (dominated by 1)
Removing Company 2 strategy 5 (dominated by 1)


2nd Company,1,4
1st Company,,
1,-74.16,1.92
2,-6.14,-0.12
3,0.61,0.00
4,-1.28,0.00
5,-2.42,1.71


# Check for saddle point

In [25]:
saddle_res = find_equilibrium(df_reduced_2)

Alpha (Maximin): 0.00
Beta (Minimax): 0.61
STATUS: No saddle point found. Mixed strategies required.


# There is no saddle point for this case, we must use mixed strategies solution

In [26]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog


def solve_mixed_strategies(df):
    """
    Solves a zero-sum game using the Simplex method via Linear Programming.
    Calculates optimal mixed strategies for both players and the game value.
    """
    # Convert DataFrame to numpy for matrix operations
    matrix = df.to_numpy()
    num_rows, num_cols = matrix.shape

    # 1. Shift matrix to positive values (K-transformation)
    # This ensures V > 0 for the linear programming solver
    min_val = matrix.min()
    k = abs(min_val) if min_val < 0 else 0
    shifted_matrix = matrix + k

    # Objective: Minimize Z = x1 + x2 + ... + xm
    c1 = np.ones(num_rows)
    # Constraints: - (A_transpose * x) <= -1
    A_ub1 = -shifted_matrix.T
    b_ub1 = -np.ones(num_cols)

    res1 = linprog(c1, A_ub=A_ub1, b_ub=b_ub1, method="highs")

    # Objective: Maximize Z = y1 + y2 + ... + yn (represented as minimize -Z)
    c2 = -np.ones(num_cols)
    # Constraints: A * y <= 1
    A_ub2 = shifted_matrix
    b_ub2 = np.ones(num_rows)

    res2 = linprog(c2, A_ub=A_ub2, b_ub=b_ub2, method="highs")

    if not res1.success or not res2.success:
        return "Error: Simplex algorithm failed to converge."

    # 2. Calculate the Game Value (V) and Probabilities
    # The solver finds Z = 1/V_star
    z_val = res1.fun
    v_star = 1 / z_val
    game_value = v_star - k

    # Calculate probabilities: p_i = x_i * V_star
    p_probabilities = res1.x * v_star
    q_probabilities = res2.x * v_star

    return {
        "game_value": game_value,
        "p_strategies": np.round(p_probabilities, 4),
        "q_strategies": np.round(q_probabilities, 4),
        "row_labels": df.index.tolist(),
        "col_labels": df.columns.tolist(),
    }


results = solve_mixed_strategies(df_reduced_2)

if isinstance(results, dict):
    print(f"Game Value (V): {results['game_value']:.4f}")

    print("\n[Company 1] Optimal Mixed Strategy:")
    for label, p in zip(results["row_labels"], results["p_strategies"]):
        if p > 0:  # Only show strategies with non-zero probability
            print(f"  -> Technology {label}: {p:.2%}")

    print("\n[Company 2] Optimal Mixed Strategy:")
    for label, q in zip(results["col_labels"], results["q_strategies"]):
        if q > 0:
            print(f"  -> Technology {label}: {q:.2%}")

    print("\nEconomic Conclusion:")
    if results["game_value"] > 0:
        print(
            f"The market conditions favor Company 1 with a net gain of {results['game_value']:.4f}."
        )
    elif results["game_value"] < 0:
        print(
            f"The market conditions favor Company 2 with a net gain of {abs(results['game_value']):.4f}."
        )
    else:
        print("The game is perfectly balanced.")
else:
    print(results)

Game Value (V): 0.2201

[Company 1] Optimal Mixed Strategy:
  -> Technology 3: 87.13%
  -> Technology 5: 12.87%

[Company 2] Optimal Mixed Strategy:
  -> Technology 1: 36.08%
  -> Technology 4: 63.92%

Economic Conclusion:
The market conditions favor Company 1 with a net gain of 0.2201.
